<a href="https://colab.research.google.com/github/Tauhid-Topu-007/Thesis-4-1/blob/main/ea-frcnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!mkdir ~/.kaggle

mkdir: cannot create directory ‘/root/.kaggle’: File exists


In [2]:

!cp kaggle.json ~/.kaggle/

cp: cannot stat 'kaggle.json': No such file or directory


In [3]:
!kaggle datasets download sharansmenon/aquarium-dataset

Dataset URL: https://www.kaggle.com/datasets/sharansmenon/aquarium-dataset
License(s): copyright-authors
aquarium-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


In [4]:
!unzip /content/aquarium-dataset.zip -d /content/

Archive:  /content/aquarium-dataset.zip
replace /content/Aquarium Combined/README.dataset.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: /content/Aquarium Combined/README.dataset.txt  
  inflating: /content/Aquarium Combined/README.roboflow.txt  
  inflating: /content/Aquarium Combined/test/IMG_2289_jpeg_jpg.rf.fe2a7a149e7b11f2313f5a7b30386e85.jpg  
  inflating: /content/Aquarium Combined/test/IMG_2301_jpeg_jpg.rf.2c19ae5efbd1f8611b5578125f001695.jpg  
  inflating: /content/Aquarium Combined/test/IMG_2319_jpeg_jpg.rf.6e20bf97d17b74a8948aa48776c40454.jpg  
  inflating: /content/Aquarium Combined/test/IMG_2347_jpeg_jpg.rf.7c71ac4b9301eb358cd4a832844dedcb.jpg  
  inflating: /content/Aquarium Combined/test/IMG_2354_jpeg_jpg.rf.396e872c7fb0a95e911806986995ee7a.jpg  
  inflating: /content/Aquarium Combined/test/IMG_2371_jpeg_jpg.rf.54505f60b6706da151c164188c305849.jpg  
  inflating: /content/Aquarium Combined/test/IMG_2379_jpeg_jpg.rf.7dc3160c937072d26d4624c6c48e904d.jpg  
  infla

In [5]:
!pip install -q pycocotools

In [6]:
# ============================================
# BLOCK 1: Imports and Setup
# ============================================

import os
import math
import time
import random
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Iterable

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision.transforms import functional as TF
from torchvision.ops import box_iou, batched_nms, generalized_box_iou
from torchvision.models import ResNet50_Weights
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.backbone_utils import resnet_fpn_backbone
from torchvision.models.detection.rpn import AnchorGenerator, RegionProposalNetwork
from torchvision.models.detection.roi_heads import RoIHeads
from torchvision.models.detection.faster_rcnn import (
    FastRCNNPredictor,
    TwoMLPHead,
)
from torchvision.models.detection.transform import GeneralizedRCNNTransform
from torchvision.ops import MultiScaleRoIAlign
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm

try:
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    HAS_COCO = True
except Exception:
    HAS_COCO = False

warnings.filterwarnings("ignore")
%matplotlib inline

print("✅ Block 1: Imports loaded successfully!")
print(f"PyTorch: {torch.__version__}")
print(f"Torchvision: {torchvision.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

✅ Block 1: Imports loaded successfully!
PyTorch: 2.11.0+cu128
Torchvision: 0.26.0+cu128
CUDA Available: True


In [7]:
# ============================================
# BLOCK 2: Configuration
# ============================================

@dataclass
class EAConfig:
    """EA-FRCNN Configuration"""

    # Dataset
    dataset_path: str = "/content/Aquarium Combined"
    img_size: int = 640
    min_size: int = 640
    max_size: int = 1024

    # Model
    num_classes: int = 8
    trainable_backbone_layers: int = 3

    # Anchors - will be adaptive
    anchor_sizes: Tuple[int, ...] = (16, 32, 64, 128, 256)
    anchor_ratios: Tuple[float, ...] = (0.33, 0.5, 1.0, 2.0, 3.0)

    # Adaptive RPN
    rpn_fg_iou_thresh: float = 0.60
    rpn_bg_iou_thresh: float = 0.30
    rpn_batch_size_per_image: int = 256
    rpn_positive_fraction: float = 0.50
    rpn_pre_nms_top_n_train: int = 4000
    rpn_pre_nms_top_n_test: int = 2000
    rpn_post_nms_top_n_train: int = 1000
    rpn_post_nms_top_n_test: int = 300
    rpn_nms_thresh: float = 0.70

    # RoI Head
    box_fg_iou_thresh: float = 0.50
    box_bg_iou_thresh: float = 0.50
    box_batch_size_per_image: int = 512
    box_positive_fraction: float = 0.25
    box_score_thresh: float = 0.05
    box_nms_thresh: float = 0.50
    box_detections_per_img: int = 100

    # Attention
    attention_reduction: int = 16

    # CIoU Loss
    ciou_weight: float = 0.50

    # Optimization
    lr: float = 0.005
    backbone_lr: float = 0.0005
    momentum: float = 0.9
    weight_decay: float = 1e-4
    epochs: int = 20
    grad_clip: float = 5.0
    amp: bool = True
    batch_size: int = 4

    # Scheduler
    scheduler_type: str = 'cosine'  # 'cosine' or 'step'

    # Device
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    seed: int = 42

    @staticmethod
    def set_seed():
        random.seed(42)
        np.random.seed(42)
        torch.manual_seed(42)
        torch.cuda.manual_seed_all(42)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

# Initialize config
config = EAConfig()
config.set_seed()

print("✅ Block 2: Configuration loaded!")
print(f"Device: {config.device}")
print(f"Dataset: {config.dataset_path}")
print(f"Image size: {config.img_size}x{config.img_size}")
print(f"Classes: {config.num_classes}")

✅ Block 2: Configuration loaded!
Device: cuda
Dataset: /content/Aquarium Combined
Image size: 640x640
Classes: 8


In [8]:
# ============================================
# BLOCK 3: Adaptive Anchor Generation
# ============================================

def extract_box_wh_from_coco(
    annotation_json: str,
    max_boxes: Optional[int] = None,
) -> np.ndarray:
    """Extract width/height from COCO annotations"""
    if not HAS_COCO:
        raise ImportError("Install pycocotools first.")

    coco = COCO(annotation_json)
    ann_ids = coco.getAnnIds(iscrowd=False)
    anns = coco.loadAnns(ann_ids)

    wh = []
    for ann in anns:
        if "bbox" not in ann:
            continue
        _, _, w, h = ann["bbox"]
        if w > 1 and h > 1:
            wh.append([w, h])

    wh = np.asarray(wh, dtype=np.float32)

    if max_boxes is not None and len(wh) > max_boxes:
        rng = np.random.default_rng(42)
        idx = rng.choice(len(wh), max_boxes, replace=False)
        wh = wh[idx]

    return wh


def kmeans_numpy(
    x: np.ndarray,
    k: int,
    seed: int = 42,
    iterations: int = 100,
) -> Tuple[np.ndarray, np.ndarray]:
    """K-means clustering implementation"""
    rng = np.random.default_rng(seed)

    if len(x) < k:
        raise ValueError(f"Need at least {k} boxes, got {len(x)}.")

    centers = x[rng.choice(len(x), k, replace=False)].copy()

    for _ in range(iterations):
        d2 = ((x[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
        labels = np.argmin(d2, axis=1)

        new_centers = []
        for j in range(k):
            members = x[labels == j]
            if len(members) == 0:
                new_centers.append(centers[j])
            else:
                new_centers.append(members.mean(axis=0))

        new_centers = np.asarray(new_centers)

        if np.allclose(new_centers, centers, atol=1e-4):
            centers = new_centers
            break

        centers = new_centers

    return centers, labels


def generate_adaptive_anchor_sizes(
    annotation_json: str,
    num_clusters: int = 5,
    max_boxes: int = 50000,
) -> List[int]:
    """Generate dataset-adaptive anchor sizes"""
    wh = extract_box_wh_from_coco(annotation_json, max_boxes=max_boxes)

    # Cluster log width/height
    x = np.log(np.maximum(wh, 1.0))
    centers, _ = kmeans_numpy(x, k=num_clusters, seed=42)

    wh_centers = np.exp(centers)
    areas = wh_centers[:, 0] * wh_centers[:, 1]

    # Convert to square scales
    scales = np.sqrt(areas)
    scales = np.clip(np.round(scales), 8, 1024).astype(int)

    return sorted(np.unique(scales).tolist())

print("✅ Block 3: Adaptive anchor generation ready!")

✅ Block 3: Adaptive anchor generation ready!


In [9]:
# ============================================
# BLOCK 4: Adaptive FPN Backbone
# ============================================

class AdaptiveFPNFusion(nn.Module):
    """
    Learned multi-scale fusion with gating mechanism
    """
    def __init__(self, channels: int = 256, levels: int = 5, reduction: int = 16):
        super().__init__()
        self.levels = levels
        hidden = max(channels // reduction, 8)

        self.proj = nn.ModuleList([
            nn.Conv2d(channels, channels, kernel_size=1, bias=False)
            for _ in range(levels)
        ])

        self.gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(channels * levels, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, levels),
            nn.Softmax(dim=1)
        )

        self.out_norm = nn.ModuleList([
            nn.GroupNorm(32, channels)
            for _ in range(levels)
        ])

    def forward(self, features: Dict[str, torch.Tensor]):
        keys = list(features.keys())
        levels = len(keys)

        projected = [self.proj[i](features[k]) for i, k in enumerate(keys)]

        # Compute gate weights
        pooled = [
            F.adaptive_avg_pool2d(x, 1).flatten(1)
            for x in projected
        ]
        descriptor = torch.cat(pooled, dim=1)
        weights = self.gate(descriptor)

        output = {}
        for i, k in enumerate(keys):
            x = projected[i]
            w = weights[:, i].view(-1, 1, 1, 1)
            y = self.out_norm[i](x + w * x)
            output[k] = y

        return output


class AdaptiveFPNBackbone(nn.Module):
    """ResNet-50 with Adaptive FPN"""
    def __init__(
        self,
        trainable_layers: int = 3,
        pretrained: bool = True,
    ):
        super().__init__()

        weights = ResNet50_Weights.DEFAULT if pretrained else None

        self.base = resnet_fpn_backbone(
            backbone_name="resnet50",
            weights=weights,
            trainable_layers=trainable_layers,
        )

        self.out_channels = self.base.out_channels
        self.adaptive_fusion = AdaptiveFPNFusion(
            channels=self.out_channels,
            levels=5,
        )

    def forward(self, x):
        features = self.base(x)
        return self.adaptive_fusion(features)

print("✅ Block 4: Adaptive FPN Backbone ready!")

✅ Block 4: Adaptive FPN Backbone ready!


In [10]:
# ============================================
# BLOCK 5: Adaptive RPN
# ============================================

class AdaptiveRegionProposalNetwork(RegionProposalNetwork):
    """
    RPN with adaptive IoU thresholds and proposal budgets
    """
    def __init__(
        self,
        anchor_generator,
        head,
        fg_iou_thresh=0.60,
        bg_iou_thresh=0.30,
        batch_size_per_image=256,
        positive_fraction=0.50,
        pre_nms_top_n=dict(training=4000, testing=2000),
        post_nms_top_n=dict(training=1000, testing=300),
        nms_thresh=0.70,
        score_thresh=0.0,
    ):
        super().__init__(
            anchor_generator=anchor_generator,
            head=head,
            fg_iou_thresh=fg_iou_thresh,
            bg_iou_thresh=bg_iou_thresh,
            batch_size_per_image=batch_size_per_image,
            positive_fraction=positive_fraction,
            pre_nms_top_n=pre_nms_top_n,
            post_nms_top_n=post_nms_top_n,
            nms_thresh=nms_thresh,
            score_thresh=score_thresh,
        )

print("✅ Block 5: Adaptive RPN ready!")

✅ Block 5: Adaptive RPN ready!


In [11]:
# ============================================
# BLOCK 6: Attention Modules
# ============================================

class ChannelAttention(nn.Module):
    """Channel attention module (CBAM style)"""
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        hidden = max(channels // reduction, 8)

        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.mlp = nn.Sequential(
            nn.Conv2d(channels, hidden, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, 1, bias=False),
        )

    def forward(self, x):
        a = self.mlp(self.avg_pool(x))
        m = self.mlp(self.max_pool(x))
        return x * torch.sigmoid(a + m)


class SpatialAttention(nn.Module):
    """Spatial attention module (CBAM style)"""
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)

    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        mx = torch.max(x, dim=1, keepdim=True).values
        attention = torch.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))
        return x * attention


class AttentionRoIFeatureExtractor(nn.Module):
    """Combined channel + spatial attention for RoI features"""
    def __init__(self, channels: int = 256, reduction: int = 16):
        super().__init__()
        self.channel_attention = ChannelAttention(channels, reduction=reduction)
        self.spatial_attention = SpatialAttention()

    def forward(self, x):
        x = self.channel_attention(x)
        x = self.spatial_attention(x)
        return x


class AttentionTwoMLPHead(nn.Module):
    """RoI head with attention before MLP"""
    def __init__(
        self,
        in_channels: int,
        resolution: int,
        representation_size: int,
        reduction: int = 16,
    ):
        super().__init__()

        self.attention = AttentionRoIFeatureExtractor(
            channels=in_channels,
            reduction=reduction,
        )

        self.flatten = nn.Flatten()

        self.fc6 = nn.Linear(
            in_channels * resolution * resolution,
            representation_size,
        )
        self.fc7 = nn.Linear(
            representation_size,
            representation_size,
        )

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.attention(x)
        x = self.flatten(x)
        x = self.relu(self.fc6(x))
        x = self.relu(self.fc7(x))
        return x

print("✅ Block 6: Attention modules ready!")

✅ Block 6: Attention modules ready!


In [12]:
# ============================================
# BLOCK 7: CIoU Loss
# ============================================

def ciou_loss(
    boxes1: torch.Tensor,
    boxes2: torch.Tensor,
    eps: float = 1e-7,
) -> torch.Tensor:
    """
    Complete IoU (CIoU) loss with center distance and aspect ratio penalty
    """
    if boxes1.numel() == 0 or boxes2.numel() == 0:
        return boxes1.sum() * 0.0

    # Intersection
    x1 = torch.max(boxes1[:, 0], boxes2[:, 0])
    y1 = torch.max(boxes1[:, 1], boxes2[:, 1])
    x2 = torch.min(boxes1[:, 2], boxes2[:, 2])
    y2 = torch.min(boxes1[:, 3], boxes2[:, 3])

    inter = (x2 - x1).clamp(min=0) * (y2 - y1).clamp(min=0)

    # Areas
    w1 = (boxes1[:, 2] - boxes1[:, 0]).clamp(min=eps)
    h1 = (boxes1[:, 3] - boxes1[:, 1]).clamp(min=eps)
    w2 = (boxes2[:, 2] - boxes2[:, 0]).clamp(min=eps)
    h2 = (boxes2[:, 3] - boxes2[:, 1]).clamp(min=eps)

    area1 = w1 * h1
    area2 = w2 * h2
    union = area1 + area2 - inter
    iou = inter / (union + eps)

    # Center distance
    c1x = (boxes1[:, 0] + boxes1[:, 2]) / 2
    c1y = (boxes1[:, 1] + boxes1[:, 3]) / 2
    c2x = (boxes2[:, 0] + boxes2[:, 2]) / 2
    c2y = (boxes2[:, 1] + boxes2[:, 3]) / 2

    center_distance = (c1x - c2x) ** 2 + (c1y - c2y) ** 2

    # Diagonal of enclosing box
    enc_x1 = torch.min(boxes1[:, 0], boxes2[:, 0])
    enc_y1 = torch.min(boxes1[:, 1], boxes2[:, 1])
    enc_x2 = torch.max(boxes1[:, 2], boxes2[:, 2])
    enc_y2 = torch.max(boxes1[:, 3], boxes2[:, 3])

    diagonal = ((enc_x2 - enc_x1) ** 2 + (enc_y2 - enc_y1) ** 2).clamp(min=eps)

    # Aspect ratio penalty
    v = (4.0 / math.pi**2) * (torch.atan(w2 / h2) - torch.atan(w1 / h1)) ** 2

    with torch.no_grad():
        alpha = v / (1.0 - iou + v + eps)

    loss = 1.0 - iou + center_distance / diagonal + alpha * v

    return loss.mean()

print("✅ Block 7: CIoU Loss ready!")

✅ Block 7: CIoU Loss ready!


In [13]:
class AttentionTwoMLPHead(nn.Module):
    def __init__(
        self,
        in_channels: int,
        resolution: int,
        representation_size: int,
        reduction: int = 16,
    ):
        super().__init__()

        self.attention = AttentionRoIFeatureExtractor(
            channels=in_channels,
            reduction=reduction,
        )

        self.flatten = nn.Flatten()

        self.fc6 = nn.Linear(
            in_channels * resolution * resolution,
            representation_size,
        )
        self.fc7 = nn.Linear(
            representation_size,
            representation_size,
        )

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.attention(x)
        x = self.flatten(x)
        x = self.relu(self.fc6(x))
        x = self.relu(self.fc7(x))
        return x


# NOTE:
# CIoU is implemented as a separate auxiliary training loss below. This avoids
# copying version-specific torchvision RoIHeads internals and is substantially
# more robust across torchvision releases.


In [14]:
# ============================================
# STEP 10: Model builder - FIXED
# ============================================

def make_anchor_generator(
    anchor_sizes: Iterable[int],
    aspect_ratios: Iterable[float],
):
    sizes = tuple((int(s),) for s in anchor_sizes)
    ratios = tuple(
        tuple(float(r) for r in aspect_ratios)
        for _ in range(len(sizes))
    )
    return AnchorGenerator(
        sizes=sizes,
        aspect_ratios=ratios,
    )


def build_ea_frcnn(
    cfg: EAConfig,
    use_adaptive_fpn: bool = True,
    use_adaptive_anchors: bool = True,
    use_adaptive_rpn: bool = True,
    use_attention: bool = True,
):
    """
    Build EA-FRCNN model with configurable components
    FIXED: num_classes=None when box_predictor is provided
    """
    # 1. Backbone
    if use_adaptive_fpn:
        backbone = AdaptiveFPNBackbone(
            trainable_layers=cfg.trainable_backbone_layers,
            pretrained=True,
        )
    else:
        backbone = resnet_fpn_backbone(
            backbone_name="resnet50",
            weights=ResNet50_Weights.DEFAULT,
            trainable_layers=cfg.trainable_backbone_layers,
        )

    # 2. Anchor Generator
    if use_adaptive_anchors:
        anchor_sizes = cfg.anchor_sizes
        aspect_ratios = cfg.anchor_ratios
    else:
        anchor_sizes = (32, 64, 128, 256, 512)
        aspect_ratios = (0.5, 1.0, 2.0)

    anchor_generator = make_anchor_generator(anchor_sizes, aspect_ratios)

    # 3. RoI pooler
    box_roi_pool = MultiScaleRoIAlign(
        featmap_names=["0", "1", "2", "3"],
        output_size=7,
        sampling_ratio=2,
    )

    # 4. Box Head
    if use_attention:
        box_head = AttentionTwoMLPHead(
            in_channels=backbone.out_channels,
            resolution=7,
            representation_size=1024,
            reduction=cfg.attention_reduction,
        )
    else:
        box_head = TwoMLPHead(
            in_channels=backbone.out_channels,
            representation_size=1024,
        )

    # 5. Box Predictor
    box_predictor = FastRCNNPredictor(1024, cfg.num_classes)

    # 6. RPN Parameters
    if use_adaptive_rpn:
        rpn_fg = cfg.rpn_fg_iou_thresh
        rpn_bg = cfg.rpn_bg_iou_thresh
        rpn_pre_train = cfg.rpn_pre_nms_top_n_train
        rpn_pre_test = cfg.rpn_pre_nms_top_n_test
        rpn_post_train = cfg.rpn_post_nms_top_n_train
        rpn_post_test = cfg.rpn_post_nms_top_n_test
        rpn_nms = cfg.rpn_nms_thresh
    else:
        rpn_fg = 0.7
        rpn_bg = 0.3
        rpn_pre_train = 2000
        rpn_pre_test = 1000
        rpn_post_train = 2000
        rpn_post_test = 1000
        rpn_nms = 0.7

    # 7. Build Model - FIXED: num_classes=None when box_predictor is provided
    model = FasterRCNN(
        backbone=backbone,
        num_classes=None,  # ← CRITICAL FIX: Must be None
        rpn_anchor_generator=anchor_generator,
        box_roi_pool=box_roi_pool,
        box_head=box_head,
        box_predictor=box_predictor,
        rpn_fg_iou_thresh=rpn_fg,
        rpn_bg_iou_thresh=rpn_bg,
        rpn_batch_size_per_image=cfg.rpn_batch_size_per_image,
        rpn_positive_fraction=cfg.rpn_positive_fraction,
        rpn_pre_nms_top_n_train=rpn_pre_train,
        rpn_pre_nms_top_n_test=rpn_pre_test,
        rpn_post_nms_top_n_train=rpn_post_train,
        rpn_post_nms_top_n_test=rpn_post_test,
        rpn_nms_thresh=rpn_nms,
        box_fg_iou_thresh=cfg.box_fg_iou_thresh,
        box_bg_iou_thresh=cfg.box_bg_iou_thresh,
        box_batch_size_per_image=cfg.box_batch_size_per_image,
        box_positive_fraction=cfg.box_positive_fraction,
        box_score_thresh=cfg.box_score_thresh,
        box_nms_thresh=cfg.box_nms_thresh,
        box_detections_per_img=cfg.box_detections_per_img,
        min_size=cfg.min_size,
        max_size=cfg.max_size,
    )

    # 8. Replace RPN with Adaptive RPN
    if use_adaptive_rpn:
        old_rpn = model.rpn
        model.rpn = AdaptiveRegionProposalNetwork(
            anchor_generator=old_rpn.anchor_generator,
            head=old_rpn.head,
            fg_iou_thresh=rpn_fg,
            bg_iou_thresh=rpn_bg,
            batch_size_per_image=cfg.rpn_batch_size_per_image,
            positive_fraction=cfg.rpn_positive_fraction,
            pre_nms_top_n={
                "training": rpn_pre_train,
                "testing": rpn_pre_test,
            },
            post_nms_top_n={
                "training": rpn_post_train,
                "testing": rpn_post_test,
            },
            nms_thresh=rpn_nms,
            score_thresh=0.0,
        )

    return model

print("✅ build_ea_frcnn fixed with num_classes=None")

✅ build_ea_frcnn fixed with num_classes=None


In [15]:
class COCODetectionDataset(torch.utils.data.Dataset):
    def __init__(
        self,
        image_root: str,
        annotation_json: str,
        train: bool = False,
    ):
        if not HAS_COCO:
            raise ImportError("Install pycocotools first.")

        self.image_root = Path(image_root)
        self.coco = COCO(annotation_json)
        self.image_ids = sorted(self.coco.getImgIds())
        self.train = train

        cats = self.coco.loadCats(self.coco.getCatIds())
        cats = sorted(cats, key=lambda x: x["id"])

        # Detection labels must be contiguous: 1..N.
        self.cat_id_to_label = {
            cat["id"]: i + 1
            for i, cat in enumerate(cats)
        }

        self.label_to_cat_id = {
            v: k for k, v in self.cat_id_to_label.items()
        }

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        info = self.coco.loadImgs(image_id)[0]

        path = self.image_root / info["file_name"]
        image = Image.open(path).convert("RGB")

        ann_ids = self.coco.getAnnIds(
            imgIds=[image_id],
            iscrowd=False,
        )
        anns = self.coco.loadAnns(ann_ids)

        boxes = []
        labels = []
        areas = []
        iscrowd = []

        width, height = image.size

        for ann in anns:
            x, y, w, h = ann["bbox"]

            x1 = max(0.0, x)
            y1 = max(0.0, y)
            x2 = min(float(width), x + max(0.0, w))
            y2 = min(float(height), y + max(0.0, h))

            if x2 <= x1 or y2 <= y1:
                continue

            if ann["category_id"] not in self.cat_id_to_label:
                continue

            boxes.append([x1, y1, x2, y2])
            labels.append(
                self.cat_id_to_label[ann["category_id"]]
            )
            areas.append((x2 - x1) * (y2 - y1))
            iscrowd.append(0)

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            areas = torch.zeros((0,), dtype=torch.float32)
            iscrowd = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)
            areas = torch.tensor(areas, dtype=torch.float32)
            iscrowd = torch.tensor(iscrowd, dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([image_id]),
            "area": areas,
            "iscrowd": iscrowd,
        }

        # Keep horizontal flip simple and box-safe.
        if self.train and random.random() < 0.5:
            image = TF.hflip(image)
            if len(boxes):
                boxes = target["boxes"]
                old_x1 = boxes[:, 0].clone()
                old_x2 = boxes[:, 2].clone()
                boxes[:, 0] = width - old_x2
                boxes[:, 2] = width - old_x1
                target["boxes"] = boxes

        image = TF.to_tensor(image)

        return image, target


def detection_collate(batch):
    images, targets = zip(*batch)
    return list(images), list(targets)


In [16]:
TRAIN_IMAGES = "/content/Aquarium Combined/train"
TRAIN_JSON = "/content/Aquarium Combined/train/_annotations.coco.json"
VAL_IMAGES = "/content/Aquarium Combined/valid"
VAL_JSON = "/content/Aquarium Combined/valid/_annotations.coco.json"

train_dataset = COCODetectionDataset(
    TRAIN_IMAGES,
    TRAIN_JSON,
    train=True,
)

val_dataset = COCODetectionDataset(
    VAL_IMAGES,
    VAL_JSON,
    train=False,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    collate_fn=detection_collate,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=detection_collate,
)


loading annotations into memory...
Done (t=0.03s)
creating index...
index created!
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!


In [17]:
@torch.no_grad()
def match_predictions_to_targets(
    pred_boxes: torch.Tensor,
    pred_labels: torch.Tensor,
    target_boxes: torch.Tensor,
    target_labels: torch.Tensor,
    iou_threshold: float = 0.30,
):
    if (
        pred_boxes.numel() == 0
        or target_boxes.numel() == 0
    ):
        return None, None

    ious = box_iou(pred_boxes, target_boxes)

    pairs_pred = []
    pairs_target = []

    for i in range(len(pred_boxes)):
        same_class = (
            target_labels == pred_labels[i]
        )

        candidate = torch.where(same_class)[0]

        if candidate.numel() == 0:
            continue

        values = ious[i, candidate]
        best_value, best_local = values.max(dim=0)
        best_target = candidate[best_local]

        if best_value >= iou_threshold:
            pairs_pred.append(i)
            pairs_target.append(best_target.item())

    if not pairs_pred:
        return None, None

    return (
        torch.tensor(
            pairs_pred,
            dtype=torch.long,
            device=pred_boxes.device,
        ),
        torch.tensor(
            pairs_target,
            dtype=torch.long,
            device=target_boxes.device,
        ),
    )


def auxiliary_ciou_loss(
    model,
    images,
    targets,
    device,
    match_iou: float = 0.30,
):
    """
    Evaluation-style auxiliary CIoU signal.

    Because torchvision detection internals differ between releases, this helper
    is intentionally isolated. It is recommended for controlled experiments and
    ablations; if you need a fully end-to-end differentiable CIoU replacement for
    Smooth L1, copy the exact RoIHeads implementation from your installed
    torchvision version and replace the box regression term there.
    """
    was_training = model.training
    model.eval()

    with torch.no_grad():
        outputs = model(images)

    if was_training:
        model.train()

    total = torch.tensor(
        0.0,
        device=device,
    )

    count = 0

    for output, target in zip(outputs, targets):
        pred_boxes = output["boxes"].to(device)
        pred_labels = output["labels"].to(device)

        gt_boxes = target["boxes"].to(device)
        gt_labels = target["labels"].to(device)

        p_idx, t_idx = match_predictions_to_targets(
            pred_boxes,
            pred_labels,
            gt_boxes,
            gt_labels,
            iou_threshold=match_iou,
        )

        if p_idx is None:
            continue

        total = total + ciou_loss(
            pred_boxes[p_idx],
            gt_boxes[t_idx],
        )
        count += 1

    if count == 0:
        return total

    return total / count


In [18]:
## Training
def train_one_epoch(
    model,
    loader,
    optimizer,
    scaler,
    device,
    epoch: int,
    cfg: EAConfig,
):
    model.train()

    running = {
        "loss_total": 0.0,
        "loss_classifier": 0.0,
        "loss_box_reg": 0.0,
        "loss_objectness": 0.0,
        "loss_rpn_box_reg": 0.0,
    }

    start = time.time()

    for step, (images, targets) in enumerate(loader, start=1):
        images = [
            img.to(device, non_blocking=True)
            for img in images
        ]

        targets = [
            {
                k: v.to(device, non_blocking=True)
                for k, v in t.items()
            }
            for t in targets
        ]

        optimizer.zero_grad(set_to_none=True)

        use_amp = (
            cfg.amp
            and device.type == "cuda"
        )

        with torch.cuda.amp.autocast(
            enabled=use_amp
        ):
            loss_dict = model(
                images,
                targets,
            )

            loss = sum(
                value
                for value in loss_dict.values()
            )

        if not torch.isfinite(loss):
            print(
                f"Skipping non-finite loss at step {step}:",
                loss.item(),
            )
            continue

        if use_amp:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                cfg.grad_clip,
            )
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                cfg.grad_clip,
            )
            optimizer.step()

        running["loss_total"] += float(loss.detach())

        for key in running:
            if key == "loss_total":
                continue
            if key in loss_dict:
                running[key] += float(
                    loss_dict[key].detach()
                )

        if step % 20 == 0:
            print(
                f"Epoch {epoch:02d} | "
                f"Step {step:04d}/{len(loader)} | "
                f"Loss {loss.item():.4f}"
            )

    n = max(len(loader), 1)

    for key in running:
        running[key] /= n

    running["epoch_time_sec"] = time.time() - start

    return running


@torch.no_grad()
def inference_predictions(
    model,
    loader,
    device,
):
    model.eval()

    outputs_all = []
    targets_all = []

    for images, targets in loader:
        images = [
            img.to(device, non_blocking=True)
            for img in images
        ]

        outputs = model(images)

        outputs_all.extend([
            {
                k: v.detach().cpu()
                for k, v in output.items()
            }
            for output in outputs
        ])

        targets_all.extend([
            {
                k: v.detach().cpu()
                for k, v in target.items()
            }
            for target in targets
        ])

    return outputs_all, targets_all


In [19]:
## 15. Detection metrics: Precision / Recall / F1
def detection_prf1(
    predictions,
    targets,
    iou_threshold: float = 0.50,
):
    tp = 0
    fp = 0
    fn = 0

    for pred, target in zip(
        predictions,
        targets,
    ):
        p_boxes = pred["boxes"]
        p_labels = pred["labels"]
        p_scores = pred["scores"]

        t_boxes = target["boxes"]
        t_labels = target["labels"]

        order = torch.argsort(
            p_scores,
            descending=True,
        )

        p_boxes = p_boxes[order]
        p_labels = p_labels[order]

        matched = set()

        for box, label in zip(
            p_boxes,
            p_labels,
        ):
            candidates = [
                j
                for j in range(len(t_boxes))
                if j not in matched
                and t_labels[j] == label
            ]

            if not candidates:
                fp += 1
                continue

            candidate_boxes = t_boxes[candidates]

            ious = box_iou(
                box.unsqueeze(0),
                candidate_boxes,
            )[0]

            best_iou, local_idx = ious.max(dim=0)

            if best_iou >= iou_threshold:
                matched.add(
                    candidates[local_idx.item()]
                )
                tp += 1
            else:
                fp += 1

        fn += len(t_boxes) - len(matched)

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = (
        2 * precision * recall
        / max(precision + recall, 1e-12)
    )

    return {
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
    }


In [20]:
## COCO mAP evaluator
def predictions_to_coco_json(
    predictions,
    dataset: COCODetectionDataset,
):
    results = []

    for pred in predictions:
        image_id = int(
            pred["image_id"].item()
        ) if pred["image_id"].numel() else None

        if image_id is None:
            continue

        boxes = pred["boxes"].numpy()
        scores = pred["scores"].numpy()
        labels = pred["labels"].numpy()

        for box, score, label in zip(
            boxes,
            scores,
            labels,
        ):
            x1, y1, x2, y2 = box
            cat_id = dataset.label_to_cat_id[
                int(label)
            ]

            results.append({
                "image_id": image_id,
                "category_id": int(cat_id),
                "bbox": [
                    float(x1),
                    float(y1),
                    float(max(0.0, x2 - x1)),
                    float(max(0.0, y2 - y1)),
                ],
                "score": float(score),
            })

    return results


def evaluate_coco(
    predictions,
    dataset: COCODetectionDataset,
    iou_type: str = "bbox",
):
    if not HAS_COCO:
        raise ImportError("Install pycocotools.")

    coco_gt = dataset.coco

    results = predictions_to_coco_json(
        predictions,
        dataset,
    )

    if len(results) == 0:
        return {
            "AP": 0.0,
            "AP50": 0.0,
            "AP75": 0.0,
            "AP_small": 0.0,
            "AP_medium": 0.0,
            "AP_large": 0.0,
        }

    coco_dt = coco_gt.loadRes(results)

    evaluator = COCOeval(
        coco_gt,
        coco_dt,
        iouType=iou_type,
    )

    evaluator.evaluate()
    evaluator.accumulate()
    evaluator.summarize()

    stats = evaluator.stats

    return {
        "AP": float(stats[0]),
        "AP50": float(stats[1]),
        "AP75": float(stats[2]),
        "AP_small": float(stats[3]),
        "AP_medium": float(stats[4]),
        "AP_large": float(stats[5]),
    }


In [21]:
# inference speed / FPS
@torch.no_grad()
def measure_fps(
    model,
    loader,
    device,
    warmup: int = 5,
    max_images: int = 50,
):
    model.eval()

    images_seen = 0

    for images, _ in loader:
        images = [
            img.to(device)
            for img in images
        ]

        _ = model(images)

        images_seen += len(images)

        if images_seen >= warmup:
            break

    if device.type == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()
    images_seen = 0

    for images, _ in loader:
        images = [
            img.to(device)
            for img in images
        ]

        _ = model(images)

        images_seen += len(images)

        if images_seen >= max_images:
            break

    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

    fps = images_seen / max(elapsed, 1e-9)

    return {
        "images": images_seen,
        "seconds": elapsed,
        "FPS": fps,
        "ms_per_image": 1000.0 / max(fps, 1e-9),
    }


In [22]:
#Complete experiment runner
def run_experiment(
    experiment_name: str,
    cfg: EAConfig,
    train_loader,
    val_loader,
    val_dataset,
    model_kwargs: Dict,
):
    print("\n" + "=" * 80)
    print("EXPERIMENT:", experiment_name)
    print("=" * 80)

    model = build_ea_frcnn(
        cfg,
        **model_kwargs,
    ).to(DEVICE)

    params = [
        p
        for p in model.parameters()
        if p.requires_grad
    ]

    optimizer = torch.optim.SGD(
        params,
        lr=cfg.lr,
        momentum=cfg.momentum,
        weight_decay=cfg.weight_decay,
    )

    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=max(cfg.epochs // 2, 1),
        gamma=0.1,
    )

    scaler = torch.cuda.amp.GradScaler(
        enabled=(
            cfg.amp
            and DEVICE.type == "cuda"
        )
    )

    history = []

    for epoch in range(1, cfg.epochs + 1):
        stats = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            scaler=scaler,
            device=DEVICE,
            epoch=epoch,
            cfg=cfg,
        )

        scheduler.step()

        stats["epoch"] = epoch
        history.append(stats)

        print(
            f"[{experiment_name}] "
            f"Epoch {epoch}: "
            f"loss={stats['loss_total']:.4f}"
        )

    predictions, targets = inference_predictions(
        model,
        val_loader,
        DEVICE,
    )

    # Attach image_id for COCO conversion.
    for pred, target in zip(
        predictions,
        targets,
    ):
        pred["image_id"] = target["image_id"]

    prf1 = detection_prf1(
        predictions,
        targets,
        iou_threshold=0.50,
    )

    coco_metrics = evaluate_coco(
        predictions,
        val_dataset,
    )

    speed = measure_fps(
        model,
        val_loader,
        DEVICE,
    )

    result = {
        "Experiment": experiment_name,
        **prf1,
        **coco_metrics,
        **speed,
    }

    return (
        model,
        pd.DataFrame(history),
        result,
    )


In [23]:
## Recommended ablation configuration
def default_ablation_plan():
    return [
        (
            "Baseline_FasterRCNN",
            dict(
                use_adaptive_fpn=False,
                use_adaptive_anchors=False,
                use_adaptive_rpn=False,
                use_attention=False,
            ),
        ),
        (
            "Adaptive_Anchors",
            dict(
                use_adaptive_fpn=False,
                use_adaptive_anchors=True,
                use_adaptive_rpn=False,
                use_attention=False,
            ),
        ),
        (
            "Adaptive_Anchors_AFPN",
            dict(
                use_adaptive_fpn=True,
                use_adaptive_anchors=True,
                use_adaptive_rpn=False,
                use_attention=False,
            ),
        ),
        (
            "Adaptive_Anchors_AFPN_ARPN",
            dict(
                use_adaptive_fpn=True,
                use_adaptive_anchors=True,
                use_adaptive_rpn=True,
                use_attention=False,
            ),
        ),
        (
            "EA_FRCNN",
            dict(
                use_adaptive_fpn=True,
                use_adaptive_anchors=True,
                use_adaptive_rpn=True,
                use_attention=True,
            ),
        ),
    ]


In [24]:
# ============================================
# COMPLETE SETUP: Define cfg and DEVICE
# ============================================

import torch
import random
import numpy as np

# 1. Define DEVICE
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {DEVICE}")

# 2. Define cfg (if not already defined)
from dataclasses import dataclass
from typing import Tuple

@dataclass
class EAConfig:
    num_classes: int = 8
    min_size: int = 640
    max_size: int = 1024
    trainable_backbone_layers: int = 3
    anchor_sizes: Tuple[int, ...] = (16, 32, 64, 128, 256)
    anchor_ratios: Tuple[float, ...] = (0.5, 1.0, 2.0)
    rpn_fg_iou_thresh: float = 0.60
    rpn_bg_iou_thresh: float = 0.30
    rpn_batch_size_per_image: int = 256
    rpn_positive_fraction: float = 0.50
    rpn_pre_nms_top_n_train: int = 4000
    rpn_pre_nms_top_n_test: int = 2000
    rpn_post_nms_top_n_train: int = 1000
    rpn_post_nms_top_n_test: int = 300
    rpn_nms_thresh: float = 0.70
    box_fg_iou_thresh: float = 0.50
    box_bg_iou_thresh: float = 0.50
    box_batch_size_per_image: int = 512
    box_positive_fraction: float = 0.25
    box_score_thresh: float = 0.05
    box_nms_thresh: float = 0.50
    box_detections_per_img: int = 100
    attention_reduction: int = 16
    ciou_weight: float = 0.50
    lr: float = 0.005
    momentum: float = 0.9
    weight_decay: float = 1e-4
    epochs: int = 20
    grad_clip: float = 5.0
    amp: bool = True
    batch_size: int = 2
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    @staticmethod
    def set_seed(seed=42):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


# 3. Create cfg instance
cfg = EAConfig(
    num_classes=8,
    epochs=20,
    batch_size=2,
)
cfg.device = DEVICE
cfg.set_seed()

# 4. Generate adaptive anchors
print("\n📐 Generating adaptive anchors...")
TRAIN_JSON = "/content/Aquarium Combined/train/_annotations.coco.json"
adaptive_sizes = generate_adaptive_anchor_sizes(TRAIN_JSON, num_clusters=5)
print(f"Adaptive anchor sizes: {adaptive_sizes}")
cfg.anchor_sizes = tuple(adaptive_sizes)

print(f"\n✅ Configuration:")
print(f"   Dataset: Aquarium")
print(f"   Classes: {cfg.num_classes}")
print(f"   Epochs: {cfg.epochs}")
print(f"   Batch Size: {cfg.batch_size}")
print(f"   Device: {cfg.device}")
print(f"   Adaptive Anchors: {cfg.anchor_sizes}")

✅ Device: cuda

📐 Generating adaptive anchors...
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Adaptive anchor sizes: [24, 54, 59, 117, 278]

✅ Configuration:
   Dataset: Aquarium
   Classes: 8
   Epochs: 20
   Batch Size: 2
   Device: cuda
   Adaptive Anchors: (24, 54, 59, 117, 278)


In [25]:
# ============================================
# Define DEVICE
# ============================================

import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {DEVICE}")

# Also update cfg
cfg.device = DEVICE

print(f"✅ cfg.device set to: {cfg.device}")

✅ Device: cuda
✅ cfg.device set to: cuda


In [26]:
# ============================================
# FIXED: run_experiment - Uses cfg.device instead of DEVICE
# ============================================

def run_experiment(
    experiment_name: str,
    cfg: EAConfig,
    train_loader,
    val_loader,
    val_dataset,
    model_kwargs: Dict,
):
    print("\n" + "=" * 80)
    print("EXPERIMENT:", experiment_name)
    print("=" * 80)

    # Use cfg.device instead of DEVICE
    device = cfg.device

    model = build_ea_frcnn(
        cfg,
        **model_kwargs,
    ).to(device)

    # Separate backbone and other parameters for different learning rates
    backbone_params = []
    other_params = []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if 'backbone' in name:
            backbone_params.append(param)
        else:
            other_params.append(param)

    optimizer = torch.optim.SGD([
        {'params': backbone_params, 'lr': cfg.lr * 0.1},
        {'params': other_params, 'lr': cfg.lr},
    ], momentum=cfg.momentum, weight_decay=cfg.weight_decay, nesterov=True)

    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=max(cfg.epochs // 2, 1),
        gamma=0.1,
    )

    scaler = torch.cuda.amp.GradScaler(
        enabled=(
            cfg.amp
            and device.type == "cuda"
        )
    )

    history = []

    for epoch in range(1, cfg.epochs + 1):
        stats = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            scaler=scaler,
            device=device,
            epoch=epoch,
            cfg=cfg,
        )

        scheduler.step()

        stats["epoch"] = epoch
        history.append(stats)

        print(
            f"[{experiment_name}] "
            f"Epoch {epoch}: "
            f"loss={stats['loss_total']:.4f}"
        )

    predictions, targets = inference_predictions(
        model,
        val_loader,
        device,
    )

    # Attach image_id for COCO conversion.
    for pred, target in zip(
        predictions,
        targets,
    ):
        pred["image_id"] = target["image_id"]

    prf1 = detection_prf1(
        predictions,
        targets,
        iou_threshold=0.50,
    )

    coco_metrics = evaluate_coco(
        predictions,
        val_dataset,
    )

    speed = measure_fps(
        model,
        val_loader,
        device,
    )

    result = {
        "Experiment": experiment_name,
        **prf1,
        **coco_metrics,
        **speed,
    }

    return (
        model,
        pd.DataFrame(history),
        result,
    )

print("✅ run_experiment fixed - uses cfg.device")

✅ run_experiment fixed - uses cfg.device


In [27]:
# ============================================
# CELL 1: Setup Device and Config
# ============================================

import torch

# Define device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {DEVICE}")

# Create config for Aquarium dataset
cfg = EAConfig(
    num_classes=8,          # Aquarium has 8 classes
    epochs=20,              # Full training (use 2 for quick test)
    batch_size=2,           # Adjust based on GPU memory
    min_size=640,
    max_size=1024,
)
cfg.device = DEVICE
cfg.set_seed()

# Generate adaptive anchors
print("\n📐 Generating adaptive anchors...")
TRAIN_JSON = "/content/Aquarium Combined/train/_annotations.coco.json"
adaptive_sizes = generate_adaptive_anchor_sizes(TRAIN_JSON, num_clusters=5)
print(f"Adaptive anchor sizes: {adaptive_sizes}")
cfg.anchor_sizes = tuple(adaptive_sizes)

print(f"\n✅ Configuration:")
print(f"   Dataset: Aquarium")
print(f"   Classes: {cfg.num_classes}")
print(f"   Epochs: {cfg.epochs}")
print(f"   Batch Size: {cfg.batch_size}")
print(f"   Device: {cfg.device}")
print(f"   Adaptive Anchors: {cfg.anchor_sizes}")

✅ Device: cuda

📐 Generating adaptive anchors...
loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
Adaptive anchor sizes: [24, 54, 59, 117, 278]

✅ Configuration:
   Dataset: Aquarium
   Classes: 8
   Epochs: 20
   Batch Size: 2
   Device: cuda
   Adaptive Anchors: (24, 54, 59, 117, 278)


In [28]:
# ============================================
# CELL 2: Create Dataloaders
# ============================================

print("\n📁 Creating dataloaders...")

# Training dataset
train_dataset = COCODetectionDataset(
    image_root="/content/Aquarium Combined/train",
    annotation_json="/content/Aquarium Combined/train/_annotations.coco.json",
    train=True,
)

# Validation dataset
val_dataset = COCODetectionDataset(
    image_root="/content/Aquarium Combined/valid",
    annotation_json="/content/Aquarium Combined/valid/_annotations.coco.json",
    train=False,
)

# Dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=0,  # Use 0 to avoid multiprocessing issues
    pin_memory=True,
    collate_fn=detection_collate,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    collate_fn=detection_collate,
)

print(f"✅ Dataloaders ready!")
print(f"   Train: {len(train_loader)} batches")
print(f"   Validation: {len(val_loader)} batches")


📁 Creating dataloaders...
loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
✅ Dataloaders ready!
   Train: 224 batches
   Validation: 64 batches


In [29]:
# ============================================
# CELL 3: Ablation Plan
# ============================================

def default_ablation_plan():
    """Ablation experiments - keep everything identical except component being tested"""
    return [
        (
            "Baseline_FasterRCNN",
            dict(
                use_adaptive_fpn=False,
                use_adaptive_anchors=False,
                use_adaptive_rpn=False,
                use_attention=False,
            ),
        ),
        (
            "Adaptive_Anchors",
            dict(
                use_adaptive_fpn=False,
                use_adaptive_anchors=True,
                use_adaptive_rpn=False,
                use_attention=False,
            ),
        ),
        (
            "Adaptive_Anchors_AFPN",
            dict(
                use_adaptive_fpn=True,
                use_adaptive_anchors=True,
                use_adaptive_rpn=False,
                use_attention=False,
            ),
        ),
        (
            "Adaptive_Anchors_AFPN_ARPN",
            dict(
                use_adaptive_fpn=True,
                use_adaptive_anchors=True,
                use_adaptive_rpn=True,
                use_attention=False,
            ),
        ),
        (
            "EA_FRCNN",
            dict(
                use_adaptive_fpn=True,
                use_adaptive_anchors=True,
                use_adaptive_rpn=True,
                use_attention=True,
            ),
        ),
    ]

print("✅ Ablation plan defined")

✅ Ablation plan defined


In [30]:
# ============================================
# FIX: Override All Problematic Classes
# Run this entire cell before experiments
# ============================================

import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================================
# FIX 1: TwoMLPHead - Correct dimensions
# ============================================

class TwoMLPHead(nn.Module):
    """
    Standard two MLP head for Faster R-CNN
    FIXED: Proper dimension calculation
    """
    def __init__(self, in_channels, representation_size):
        super().__init__()
        # IMPORTANT: in_channels * 7 * 7 (RoI pooled size is 7x7)
        # For ResNet50 FPN: in_channels = 256
        self.fc6 = nn.Linear(in_channels * 7 * 7, representation_size)
        self.fc7 = nn.Linear(representation_size, representation_size)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = torch.flatten(x, 1)
        x = self.relu(self.fc6(x))
        x = self.relu(self.fc7(x))
        return x


# ============================================
# FIX 2: AttentionTwoMLPHead - Correct dimensions
# ============================================

class AttentionTwoMLPHead(nn.Module):
    """
    RoI head with attention before MLP
    FIXED: Proper dimension calculation
    """
    def __init__(
        self,
        in_channels: int,
        resolution: int,
        representation_size: int,
        reduction: int = 16,
    ):
        super().__init__()

        self.attention = AttentionRoIFeatureExtractor(
            channels=in_channels,
            reduction=reduction,
        )

        self.flatten = nn.Flatten()

        # IMPORTANT FIX: in_channels * resolution * resolution
        # For ResNet50 FPN: in_channels = 256, resolution = 7
        self.fc6 = nn.Linear(in_channels * resolution * resolution, representation_size)
        self.fc7 = nn.Linear(representation_size, representation_size)

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.attention(x)
        x = self.flatten(x)
        x = self.relu(self.fc6(x))
        x = self.relu(self.fc7(x))
        return x


# ============================================
# FIX 3: AdaptiveFPNFusion - Correct dimensions
# ============================================

class AdaptiveFPNFusion(nn.Module):
    """
    Learned multi-scale fusion with gating mechanism
    FIXED: Proper dimension handling
    """
    def __init__(self, channels: int = 256, levels: int = 4, reduction: int = 16):
        super().__init__()
        self.levels = levels
        hidden = max(channels // reduction, 8)

        self.proj = nn.ModuleList([
            nn.Conv2d(channels, channels, kernel_size=1, bias=False)
            for _ in range(levels)
        ])

        # FIX: Gate input size is channels * levels
        self.gate = nn.Sequential(
            nn.Linear(channels * levels, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, levels),
            nn.Softmax(dim=1)
        )

        self.out_norm = nn.ModuleList([
            nn.GroupNorm(32, channels)
            for _ in range(levels)
        ])

    def forward(self, features: Dict[str, torch.Tensor]):
        keys = list(features.keys())
        levels = len(keys)

        # Project features
        projected = [self.proj[i](features[k]) for i, k in enumerate(keys)]

        # Global pooling to get descriptors
        pooled = [
            F.adaptive_avg_pool2d(x, 1).flatten(1)  # Shape: [batch, channels]
            for x in projected
        ]

        # Concatenate descriptors
        descriptor = torch.cat(pooled, dim=1)  # Shape: [batch, channels * levels]

        # Compute weights
        weights = self.gate(descriptor)  # Shape: [batch, levels]

        # Apply weights to features
        output = {}
        for i, k in enumerate(keys):
            x = projected[i]
            w = weights[:, i].view(-1, 1, 1, 1)
            y = self.out_norm[i](x + w * x)
            output[k] = y

        return output


# ============================================
# FIX 4: AdaptiveFPNBackbone - Use 4 levels
# ============================================

class AdaptiveFPNBackbone(nn.Module):
    """ResNet-50 with Adaptive FPN"""
    def __init__(
        self,
        trainable_layers: int = 3,
        pretrained: bool = True,
    ):
        super().__init__()

        weights = ResNet50_Weights.DEFAULT if pretrained else None

        self.base = resnet_fpn_backbone(
            backbone_name="resnet50",
            weights=weights,
            trainable_layers=trainable_layers,
        )

        self.out_channels = self.base.out_channels

        # FIX: FPN has 4 levels: '0', '1', '2', '3'
        self.adaptive_fusion = AdaptiveFPNFusion(
            channels=self.out_channels,
            levels=4,  # FPN has 4 levels
        )

    def forward(self, x):
        features = self.base(x)
        return self.adaptive_fusion(features)


# ============================================
# FIX 5: build_ea_frcnn - num_classes=None
# ============================================

def build_ea_frcnn(
    cfg: EAConfig,
    use_adaptive_fpn: bool = True,
    use_adaptive_anchors: bool = True,
    use_adaptive_rpn: bool = True,
    use_attention: bool = True,
):
    # 1. Backbone
    if use_adaptive_fpn:
        backbone = AdaptiveFPNBackbone(
            trainable_layers=cfg.trainable_backbone_layers,
            pretrained=True,
        )
    else:
        backbone = resnet_fpn_backbone(
            backbone_name="resnet50",
            weights=ResNet50_Weights.DEFAULT,
            trainable_layers=cfg.trainable_backbone_layers,
        )

    # 2. Anchor Generator
    if use_adaptive_anchors:
        anchor_sizes = cfg.anchor_sizes
        aspect_ratios = cfg.anchor_ratios
    else:
        anchor_sizes = (32, 64, 128, 256, 512)
        aspect_ratios = (0.5, 1.0, 2.0)

    anchor_generator = AnchorGenerator(
        sizes=tuple((int(s),) for s in anchor_sizes),
        aspect_ratios=tuple(tuple(float(r) for r in aspect_ratios) for _ in range(len(anchor_sizes)))
    )

    # 3. RoI pooler
    box_roi_pool = MultiScaleRoIAlign(
        featmap_names=["0", "1", "2", "3"],
        output_size=7,
        sampling_ratio=2,
    )

    # 4. Box Head
    if use_attention:
        box_head = AttentionTwoMLPHead(
            in_channels=backbone.out_channels,
            resolution=7,
            representation_size=1024,
            reduction=cfg.attention_reduction,
        )
    else:
        box_head = TwoMLPHead(
            in_channels=backbone.out_channels,
            representation_size=1024,
        )

    # 5. Box Predictor
    box_predictor = FastRCNNPredictor(1024, cfg.num_classes)

    # 6. RPN Parameters
    if use_adaptive_rpn:
        rpn_fg = cfg.rpn_fg_iou_thresh
        rpn_bg = cfg.rpn_bg_iou_thresh
        rpn_pre_train = cfg.rpn_pre_nms_top_n_train
        rpn_pre_test = cfg.rpn_pre_nms_top_n_test
        rpn_post_train = cfg.rpn_post_nms_top_n_train
        rpn_post_test = cfg.rpn_post_nms_top_n_test
        rpn_nms = cfg.rpn_nms_thresh
    else:
        rpn_fg = 0.7
        rpn_bg = 0.3
        rpn_pre_train = 2000
        rpn_pre_test = 1000
        rpn_post_train = 2000
        rpn_post_test = 1000
        rpn_nms = 0.7

    # 7. Build Model - FIXED: num_classes=None
    model = FasterRCNN(
        backbone=backbone,
        num_classes=None,  # ← CRITICAL FIX
        rpn_anchor_generator=anchor_generator,
        box_roi_pool=box_roi_pool,
        box_head=box_head,
        box_predictor=box_predictor,
        rpn_fg_iou_thresh=rpn_fg,
        rpn_bg_iou_thresh=rpn_bg,
        rpn_batch_size_per_image=cfg.rpn_batch_size_per_image,
        rpn_positive_fraction=cfg.rpn_positive_fraction,
        rpn_pre_nms_top_n_train=rpn_pre_train,
        rpn_pre_nms_top_n_test=rpn_pre_test,
        rpn_post_nms_top_n_train=rpn_post_train,
        rpn_post_nms_top_n_test=rpn_post_test,
        rpn_nms_thresh=rpn_nms,
        box_fg_iou_thresh=cfg.box_fg_iou_thresh,
        box_bg_iou_thresh=cfg.box_bg_iou_thresh,
        box_batch_size_per_image=cfg.box_batch_size_per_image,
        box_positive_fraction=cfg.box_positive_fraction,
        box_score_thresh=cfg.box_score_thresh,
        box_nms_thresh=cfg.box_nms_thresh,
        box_detections_per_img=cfg.box_detections_per_img,
        min_size=cfg.min_size,
        max_size=cfg.max_size,
    )

    # 8. Replace RPN with Adaptive RPN
    if use_adaptive_rpn:
        old_rpn = model.rpn
        model.rpn = AdaptiveRegionProposalNetwork(
            anchor_generator=old_rpn.anchor_generator,
            head=old_rpn.head,
            fg_iou_thresh=rpn_fg,
            bg_iou_thresh=rpn_bg,
            batch_size_per_image=cfg.rpn_batch_size_per_image,
            positive_fraction=cfg.rpn_positive_fraction,
            pre_nms_top_n={
                "training": rpn_pre_train,
                "testing": rpn_pre_test,
            },
            post_nms_top_n={
                "training": rpn_post_train,
                "testing": rpn_post_test,
            },
            nms_thresh=rpn_nms,
            score_thresh=0.0,
        )

    return model


print("✅ All fixes applied successfully!")
print("   - TwoMLPHead: Fixed dimensions (in_channels * 7 * 7)")
print("   - AttentionTwoMLPHead: Fixed dimensions (in_channels * resolution * resolution)")
print("   - AdaptiveFPNFusion: Fixed gate input (channels * levels)")
print("   - AdaptiveFPNBackbone: Using 4 levels")
print("   - build_ea_frcnn: num_classes=None")

✅ All fixes applied successfully!
   - TwoMLPHead: Fixed dimensions (in_channels * 7 * 7)
   - AttentionTwoMLPHead: Fixed dimensions (in_channels * resolution * resolution)
   - AdaptiveFPNFusion: Fixed gate input (channels * levels)
   - AdaptiveFPNBackbone: Using 4 levels
   - build_ea_frcnn: num_classes=None


In [ ]:
# ============================================
# NOW RUN EXPERIMENTS AGAIN
# ============================================

print("\n" + "="*80)
print("RUNNING EXPERIMENTS WITH ALL FIXES")
print("="*80)

all_results = []
all_histories = {}
trained_models = {}

# Quick mode: 2 epochs
QUICK_MODE = True

if QUICK_MODE:
    print("\n🚀 QUICK MODE: 2 epochs")
    cfg.epochs = 2
    experiments = [
        ("Baseline_FasterRCNN", dict(
            use_adaptive_fpn=False,
            use_adaptive_anchors=False,
            use_adaptive_rpn=False,
            use_attention=False,
        )),
        ("EA_FRCNN", dict(
            use_adaptive_fpn=True,
            use_adaptive_anchors=True,
            use_adaptive_rpn=True,
            use_attention=True,
        )),
    ]
else:
    print("\n📊 FULL ABLATION: 20 epochs")
    cfg.epochs = 20
    experiments = default_ablation_plan()

for name, flags in experiments:
    print(f"\n{'='*80}")
    print(f"Running: {name}")
    print(f"{'='*80}")

    try:
        model, history, result = run_experiment(
            experiment_name=name,
            cfg=cfg,
            train_loader=train_loader,
            val_loader=val_loader,
            val_dataset=val_dataset,
            model_kwargs=flags,
        )

        trained_models[name] = model
        all_histories[name] = history
        all_results.append(result)

        print(f"\n✅ {name} completed!")
        if 'AP' in result:
            print(f"   AP: {result['AP']:.4f}")
        if 'AP50' in result:
            print(f"   AP50: {result['AP50']:.4f}")

    except Exception as e:
        print(f"\n❌ {name} failed: {e}")
        import traceback
        traceback.print_exc()
        continue

# Results
if all_results:
    results_df = pd.DataFrame(all_results)
    print("\n" + "="*80)
    print("RESULTS SUMMARY")
    print("="*80)
    display_cols = ["Experiment", "AP", "AP50", "AP75", "Precision", "Recall", "F1"]
    print(results_df[display_cols].to_string(index=False))
    results_df.to_csv("aquarium_final_results.csv", index=False)
    print("\n✅ Results saved to aquarium_final_results.csv")


RUNNING EXPERIMENTS WITH ALL FIXES

🚀 QUICK MODE: 2 epochs

Running: Baseline_FasterRCNN

EXPERIMENT: Baseline_FasterRCNN
